# AI Research Foundations Multilingual Tokenization Challenge

## Can you build the most efficient multilingual tokenizer?

You have six languages, one vocabulary, and a budget of 10,000 tokens.

The full overview, dataset description, rules, scoring formula and submission
steps are in the [competition README](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge).

---

## What This Notebook Provides

This notebook gives you a starting point for the competition:

* Code to load and explore the **training and validation data**.
* A **standard BPE baseline** that defines the reference score.
* A **balanced BPE baseline** showing how one change to the data strategy affects multilingual tokenization.
* The official **validation scoring function**.
* Per-language results for comparing tokenizer performance.
* A **submission checker** for validating your final tokenizer.
* Code to export your tokenizer as `tokenizer.json`.

The baselines are starting points, not solutions.

# Getting Started

Install the pinned competition runtime and download the submission checker.
`tokenizers` must be exactly `0.22.1`, because that is the version official
evaluation uses.

In [ ]:
# Colab and Kaggle need the pinned competition runtime installed. A local
# environment created with `uv sync --dev` already has it, and uv virtual
# environments do not ship pip, so this cell installs only what is missing.
from importlib.metadata import PackageNotFoundError, version

TOKENIZERS_VERSION = "0.22.1"


def needs_install(package, exact=None):
    """Return True when a package is absent or not at the required version.

    Args:
        package: Distribution name to look up.
        exact: Version that must match exactly, or None for any version.

    Returns:
        True when pip should install the package.
    """
    try:
        found = version(package)
    except PackageNotFoundError:
        return True
    return exact is not None and found != exact


missing = []
if needs_install("tokenizers", TOKENIZERS_VERSION):
    missing.append(f"tokenizers=={TOKENIZERS_VERSION}")
for package, requirement in (("datasets", "datasets>=4.0,<5"),
                             ("pandas", "pandas"),
                             ("matplotlib", "matplotlib")):
    if needs_install(package):
        missing.append(requirement)

if missing:
    packages = " ".join(missing)
    !pip install -q {packages}
else:
    print("Dependencies already available at the required versions.")

In [ ]:
# Competition configuration.
GITHUB_REPO = "aims-ai-research-foundations/airf-multilingual-tokenizer-challenge"
GITHUB_BRANCH = "main"
HF_DATASET = "Similoluwa/african-multilingual-tokenizer-challenge"
HF_REVISION = "v1.0.0"

LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {
    "en": "English",
    "fr": "French",
    "ha": "Hausa",
    "sw": "Swahili",
    "yo": "Yoruba",
    "am": "Amharic",
}
MAX_VOCAB_SIZE = 10_000

RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}"

In [ ]:
# Download the one helper file you need. It checks your finished tokenizer.
!wget -q -O utils.py $RAW_URL/starter/utils.py

import urllib.request
from collections import defaultdict

import matplotlib.pyplot as plt
import pandas as pd
import tokenizers
from datasets import load_dataset
from tokenizers import (Tokenizer, decoders, models, normalizers,
                        pre_tokenizers, trainers)

from utils import profile_submission

print("tokenizers version:", tokenizers.__version__)

## Load the Dataset

Let's start by loading the competition data and taking a look at what we're working with.

Before building a tokenizer, inspect the size of the dataset, the distribution of languages, and a few examples from each language.

Understanding your data is part of understanding your tokenizer.

In [ ]:
def load_competition_data(split):
    """Load one public competition split as a pandas DataFrame.

    Args:
        split: Either "train" or "validation".

    Returns:
        A DataFrame with a `language` column and a `text` column.
    """
    if split not in {"train", "validation"}:
        raise ValueError("only the train and validation splits are public")
    dataset = load_dataset(HF_DATASET, split=split, revision=HF_REVISION)
    frame = dataset.to_pandas()[["language", "text"]]
    return frame.reset_index(drop=True)


train = load_competition_data("train")
validation = load_competition_data("validation")

print(f"Train rows:      {len(train):,}")
print(f"Validation rows: {len(validation):,}")

In [ ]:
# Rows are balanced across languages, but characters are not.
summary = train.assign(characters=train.text.str.len()).groupby("language").agg(
    rows=("text", "size"),
    characters=("characters", "sum"),
    mean_length=("characters", "mean"),
)
summary["share_of_characters"] = summary.characters / summary.characters.sum()
summary.index = [LANGUAGE_NAMES[language] for language in summary.index]
summary.sort_values("characters", ascending=False).round(
    {"mean_length": 1, "share_of_characters": 3}
)

In [ ]:
# Every language contributes the same number of rows but not the same amount of
# text, and the tokenizer learns its vocabulary from characters, not from rows.
sizes = summary.characters.sort_values() / 1e6

figure, axes = plt.subplots(figsize=(7.2, 3.4))
bars = axes.barh(sizes.index, sizes.values, height=0.62, color="#4878a8")
axes.bar_label(bars, fmt="%.2fM", padding=5, fontsize=9, color="#444444")

axes.set_xlabel("Characters in the training split (millions)")
axes.set_title("Same rows per language, different amounts of text", pad=12)
axes.set_xlim(0, sizes.max() * 1.18)
axes.xaxis.grid(True, color="#e8e8e8", linewidth=0.8)
axes.set_axisbelow(True)
axes.tick_params(axis="y", length=0)
for edge in ("top", "right", "left"):
    axes.spines[edge].set_visible(False)

plt.tight_layout()
plt.show()

spread = summary.characters.max() / summary.characters.min()
print(f"Largest language carries {spread:.2f}x the characters of the smallest.")

In [ ]:
# One example per language, so you can see what you are tokenizing.
for language in LANGUAGES:
    text = train.loc[train.language == language, "text"].iloc[0]
    print(f"[{language}] {LANGUAGE_NAMES[language]}")
    print(f"     {text[:120]}")

# Baseline 1: Standard BPE

## Start Simple

We begin with **Byte Pair Encoding (BPE)**, a widely used subword tokenization algorithm.

BPE starts with small units and repeatedly merges frequently occurring pairs to build a vocabulary of useful subwords.

For our first baseline, we keep things simple:

* Train directly on the provided multilingual training data.
* Use a maximum vocabulary size of **10,000 tokens**.
* Apply the same tokenizer across all six languages.

This tokenizer defines the official competition baseline.

$$
\boxed{\text{Baseline Score} = 1.000}
$$

Every experiment can now be compared against a common starting point.

## Train Baseline 1

Run the cells below to train the standard BPE tokenizer.

Once training is complete, evaluate it on the validation set.

In [ ]:
def train_baseline_bpe(texts, vocab_size=MAX_VOCAB_SIZE):
    """Train the official byte level BPE baseline.

    Byte level pre-tokenization with `byte_fallback` means every possible
    character can be encoded, so nothing ever becomes an unknown token. NFC
    normalization keeps canonically equivalent text consistent, which matters
    for the diacritics in Yoruba.

    Args:
        texts: An iterable of training strings.
        vocab_size: The vocabulary budget, at most 10,000.

    Returns:
        A trained `tokenizers.Tokenizer`.
    """
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]", byte_fallback=True))
    tokenizer.normalizer = normalizers.NFC()
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False,
                                                       use_regex=True)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=["[UNK]"],
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        show_progress=True,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer, length=len(texts))
    return tokenizer


standard_bpe = train_baseline_bpe(train.text.tolist())
print(f"Learned vocabulary: "
      f"{standard_bpe.get_vocab_size(with_added_tokens=True):,} / {MAX_VOCAB_SIZE:,}")

## Evaluate Baseline 1

Let's see how our first tokenizer performs.

Look at both the **overall score** and the **per-language scores**.

The overall score gives you a single measure of performance. The per-language scores show where the tokenizer performs well and where there may be room for improvement.

**We now have something to beat.**

In [ ]:
def count_characters(text):
    """Count Unicode code points in a string, excluding whitespace."""
    return sum(1 for character in text if not character.isspace())


def fertility(tokenizer, data):
    """Compute tokens per non-whitespace character for every language.

    Args:
        tokenizer: A trained `tokenizers.Tokenizer`.
        data: A DataFrame with `language` and `text` columns.

    Returns:
        A dictionary mapping each language code to its fertility.
    """
    tokens = defaultdict(int)
    characters = defaultdict(int)
    encodings = tokenizer.encode_batch(data.text.tolist(), add_special_tokens=False)
    for language, text, encoding in zip(data.language, data.text, encodings):
        tokens[language] += len(encoding.ids)
        characters[language] += count_characters(text)
    return {language: tokens[language] / characters[language] for language in LANGUAGES}


def load_official_baseline():
    """Download the official baseline tokenizer used to normalize scores."""
    urllib.request.urlretrieve(f"{RAW_URL}/submissions/baseline/tokenizer.json",
                               "baseline_tokenizer.json")
    return Tokenizer.from_file("baseline_tokenizer.json")


baseline_fertility = fertility(load_official_baseline(), validation)

In [ ]:
def score(tokenizer, data=validation, name="candidate"):
    """Score a tokenizer with the official competition metric.

    Args:
        tokenizer: A trained `tokenizers.Tokenizer`.
        data: Labelled text to score on, by default the validation split.
        name: A label used when the result is printed.

    Returns:
        A dictionary with the score, per language fertility and normalized
        fertility, and the vocabulary size.
    """
    measured = fertility(tokenizer, data)
    normalized = {
        language: measured[language] / baseline_fertility[language]
        for language in LANGUAGES
    }
    return {
        "name": name,
        "score": sum(normalized.values()) / len(LANGUAGES),
        "fertility": measured,
        "normalized": normalized,
        "vocab_size": tokenizer.get_vocab_size(with_added_tokens=True),
    }


def show_score(result):
    """Print one scored result as a per language table."""
    print(f"{result['name']}: score {result['score']:.4f}   "
          f"vocabulary {result['vocab_size']:,} / {MAX_VOCAB_SIZE:,}")
    for language in LANGUAGES:
        print(f"  {LANGUAGE_NAMES[language]:<9} "
              f"fertility {result['fertility'][language]:.4f}   "
              f"normalized {result['normalized'][language]:.4f}")


standard_result = score(standard_bpe, name="Baseline 1: Standard BPE")
show_score(standard_result)

# Baseline 2: Balanced BPE

## Change One Thing

Our first tokenizer learns its vocabulary directly from the training corpus.

But multilingual datasets are rarely perfectly balanced. The competition data holds the same number of rows per language, yet passages differ in length, so some languages contribute far more characters than others. Those languages have a greater influence on which tokens make it into our limited vocabulary.

So let's change **one thing**.

For Baseline 2, we keep:

* The same **BPE algorithm**.
* The same **10,000-token vocabulary budget**.
* The same **six languages**.

But we change how the training data is sampled.

Instead of allowing languages with more text to dominate vocabulary learning, we repeat rows so that every language contributes the same number of characters.

## Train Baseline 2

Run the cells below to construct a language-balanced training corpus and train the second BPE tokenizer.

This gives us a controlled experiment:

**Same algorithm. Same vocabulary budget. Different data strategy.**

In [ ]:
def build_balanced_corpus(data):
    """Repeat rows so every language contributes the same number of characters.

    Each language is cycled through until it reaches the character budget of
    the language that currently contributes the most text. No rows are
    discarded, so nothing is lost from the smaller languages.

    Args:
        data: A DataFrame with `language` and `text` columns.

    Returns:
        A list of training strings balanced by character count.
    """
    texts_by_language = {
        language: data.loc[data.language == language, "text"].tolist()
        for language in LANGUAGES
    }
    budget = max(sum(len(text) for text in texts)
                 for texts in texts_by_language.values())

    corpus = []
    for texts in texts_by_language.values():
        used = 0
        index = 0
        while used < budget:
            text = texts[index % len(texts)]
            corpus.append(text)
            used += len(text)
            index += 1
    return corpus


balanced_corpus = build_balanced_corpus(train)
print(f"Rows after balancing: {len(balanced_corpus):,} "
      f"(original: {len(train):,})")

In [ ]:
balanced_bpe = train_baseline_bpe(balanced_corpus)
print(f"Learned vocabulary: "
      f"{balanced_bpe.get_vocab_size(with_added_tokens=True):,} / {MAX_VOCAB_SIZE:,}")

## Compare the Baselines

Now evaluate Baseline 2 using exactly the same validation set and metric.

| Tokenizer    |           Score |
| ------------ | --------------: |
| Standard BPE |       **1.000** |
| Balanced BPE | **Your result** |

Don't stop at the overall score.

Look at the results for each language:

* Which languages improved?
* Which languages became worse?
* How large were the changes?
* Did the overall score improve?

This simple experiment demonstrates an important idea:

**Tokenizer performance depends not only on the algorithm, but also on how the vocabulary is learned from the data.**

In [ ]:
balanced_result = score(balanced_bpe, name="Baseline 2: Balanced BPE")
show_score(balanced_result)

In [ ]:
# Both baselines side by side, with the per language detail.
comparison = pd.DataFrame(
    [
        {"tokenizer": result["name"],
         "score": round(result["score"], 4),
         "vocabulary": result["vocab_size"],
         **{language: round(result["normalized"][language], 4)
            for language in LANGUAGES}}
        for result in (standard_result, balanced_result)
    ]
).set_index("tokenizer")

comparison

# Your Turn

## Can You Beat the Baselines?

We changed one design choice and measured exactly what it did, language by language.

There are many more possibilities.

Here are some directions you could explore:

* **Tokenizer algorithm:** Would a different subword algorithm perform better?
* **Language balance:** Is equal sampling the best way to allocate the vocabulary?
* **Normalization:** How should Unicode, casing, punctuation, whitespace, and different scripts be handled?
* **Pre-tokenization:** How should text be split before vocabulary learning begins?
* **Vocabulary design:** How can you make better use of the 10,000-token budget?
* **Hyperparameters:** How do tokenizer training settings affect the vocabulary you learn?
* **Combinations:** Can several individually useful ideas work even better together?

You do not need to try everything.

**Pick an idea → make one change → measure the result → keep what works.**

And remember to look beyond the final score. The per-language results can tell you what your tokenizer is actually changing.

# Prepare Your Submission

Found something that beats the baselines?

Export your final tokenizer as `tokenizer.json`, then run the official submission checker.

The checker verifies that your tokenizer:

* Loads correctly.
* Has no more than **10,000 tokens**.
* Can tokenize text from all six languages.
* Meets the required submission format.

Passing the checker means your tokenizer is ready to submit.

In [ ]:
# Keep whichever tokenizer scored lower, then check it.
best_result = min(standard_result, balanced_result, key=lambda item: item["score"])
best = standard_bpe if best_result is standard_result else balanced_bpe

best.save("tokenizer.json", pretty=True)
print(f"Saved {best_result['name']} with score {best_result['score']:.4f}\n")

report = profile_submission("tokenizer.json", data=validation)

## Final Checklist

Before submitting, make sure:

* Your tokenizer has been evaluated on the validation set.
* You have checked its performance across all six languages.
* Your vocabulary contains no more than **10,000 tokens**.
* Your final file is named `tokenizer.json`.
* Your tokenizer passes the submission checker.

Your local validation score is for experimentation.

Your position on the leaderboard is determined using the **hidden test set**.

**Submit your tokenizer, see where you rank, and keep improving.**

# References

This challenge builds on concepts introduced in the **AI Research Foundations** learning path:

* **Course 01: Build Your Own Small Language Model** introduces the language model pipeline and where tokenization fits into building a language model.
* **Course 02: Represent Your Language Data** explores how text is represented for language models, including tokenization and vocabulary construction.

**AI Research Foundations Learning Path:**
https://www.skills.google/paths/3135